In [1]:
# ================================
# 2️⃣ Imports
# ================================
import json
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# ================================
# 3️⃣ Load Dataset
# ================================
file_path = "/content/OS_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

# ================================
# 4️⃣ Split 3-shot + test
# ================================
few_shot = data[:3]     # ONLY first 3 samples
test_data = data[3:]

print("Few-shot examples:", len(few_shot))
print("Test samples:", len(test_data))

# ================================
# 5️⃣ Labels
# ================================
LABELS = sorted(list(set([ex["output"].strip() for ex in data])))
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

print("Labels:", LABELS)

# ================================
# 6️⃣ Dataset Class
# ================================
class RelationDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        inputs = self.tokenizer(
            ex["input"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        label = torch.tensor(LABEL2ID[ex["output"].strip()])
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item['labels'] = label
        return item

# ================================
# 7️⃣ Tokenizer and Model
# ================================
MODEL_NAME = "roberta-base"  # small and fast
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ================================
# 8️⃣ Prepare DataLoaders
# ================================
train_dataset = RelationDataset(few_shot, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=3, shuffle=True)

# ================================
# 9️⃣ Training (Very Quick for 3-shot)
# ================================
optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(5):  # very small dataset
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} - Loss: {loss.item():.4f}")

# ================================
# 🔟 Prediction
# ================================
model.eval()
y_true = []
y_pred = []
results = []

with torch.no_grad():
    for ex in tqdm(test_data):
        inputs = tokenizer(
            ex["input"],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        ).to(device)
        outputs = model(**inputs)
        pred_id = outputs.logits.argmax(dim=1).item()
        pred_label = ID2LABEL[pred_id]

        y_true.append(ex["output"].strip())
        y_pred.append(pred_label)

        results.append({
            "input": ex["input"],
            "ground_truth": ex["output"].strip(),
            "predicted": pred_label
        })

# ================================
# 1️⃣1️⃣ Save Results
# ================================
output_file = "/content/roberta_3shot_results.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

from google.colab import files
files.download(output_file)
print(f"\n✅ Results saved to: {output_file}")

# ================================
# 1️⃣2️⃣ Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== ROBERTA 3-SHOT RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_true, y_pred))

# ================================
# 1️⃣3️⃣ Show Examples
# ================================
for i in range(min(5, len(results))):
    print("\n--- Example ---")
    print("Input :", results[i]["input"])
    print("GT    :", results[i]["ground_truth"])
    print("Pred  :", results[i]["predicted"])

Total samples: 1957
Few-shot examples: 3
Test samples: 1954
Labels: ['based_on', 'implements', 'improves', 'no_relation', 'part_of', 'used_for']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

100%|██████████| 1954/1954 [00:19<00:00, 102.72it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Results saved to: /content/roberta_3shot_results.json

===== ROBERTA 3-SHOT RESULTS =====
Accuracy : 0.4672
Precision: 0.2183
Recall   : 0.4672
F1 Score : 0.2976

===== Classification Report =====
              precision    recall  f1-score   support

    based_on       0.00      0.00      0.00        70
  implements       0.00      0.00      0.00        99
    improves       0.00      0.00      0.00        34
 no_relation       0.47      1.00      0.64       913
     part_of       0.00      0.00      0.00       557
    used_for       0.00      0.00      0.00       281

    accuracy                           0.47      1954
   macro avg       0.08      0.17      0.11      1954
weighted avg       0.22      0.47      0.30      1954


===== Confusion Matrix =====
[[  0   0   0  70   0   0]
 [  0   0   0  99   0   0]
 [  0   0   0  34   0   0]
 [  0   0   0 913   0   0]
 [  0   0   0 557   0   0]
 [  0   0   0 281   0   0]]

--- Example ---
Input : Sentence: NTFS (New Technology File Sys

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
